## Offline PyTorch TCN vs logged `model_out_nmpkg_raw`

Loads a runtime `.npz` (same style as `analysis.ipynb`) and the IK–ID checkpoint `runs/0504_ik_id_knee_addbiomech_balanced/best_model.pt`.

**Pipeline match:** causal LPF on angle / velocity matches `CascadeUni` (`controllers/cascade.py`): same 1st-order-section stack as on the robot, with cutoffs read from `cfg/final.yaml` by default.

**PyTorch output:** raw network predictions are passed through a **6 Hz, 4th-order causal** output LPF (same structure as `infer_out_lpf` in `cascade.py`; defaults match `out_lpf_hz` / `out_lpf_order` in `cfg/final.yaml`).

**Compare:** filtered **PyTorch** Nm/kg vs logged **`model_out_nmpkg`** (device output LPF) and **`model_out_nmpkg_raw`** (pre-output-LPF TRT). Older `.npz` files without `model_out_nmpkg_raw` still run; re-record with an updated `main_knee.py` if you need that trace.

Set `FILE_PATH` and `CKPT_PATH` in the next cell if your layout differs.

In [4]:
# --- paths (edit if needed) ---
from pathlib import Path
import io
import inspect
import os
import sys

import numpy as np
import torch
import yaml
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Image

# Repository roots
PROJECT_ROOT = Path("/home/metamobility3/Jinwoo/os_kinetics")
CTRL_ROOT = PROJECT_ROOT / "knee-exo-ctrl"
CFG_PATH = CTRL_ROOT / "cfg" / "final.yaml"
CKPT_PATH = PROJECT_ROOT / "runs" / "0504_ik_id_knee_addbiomech_balanced" / "best_model.pt"
FILE_PATH = "/home/metamobility3/Jinwoo/os_kinetics/knee_0506_gain0p12_ilseung_2.npz"

sys.path.insert(0, str(PROJECT_ROOT))
from model import TCN  # noqa: E402


def _tcn_ctor_kwargs(cfg: dict) -> dict:
    allowed = {k for k in inspect.signature(TCN.__init__).parameters if k != "self"}
    return {k: v for k, v in cfg.items() if k in allowed}


class _CausalLowPass:
    """Same streaming LPF as knee `CascadeUni` (cascaded 1st-order sections)."""

    def __init__(self, fs_hz: float, cutoff_hz: float, order: int = 4):
        self.fs_hz = float(fs_hz)
        self.cutoff_hz = float(cutoff_hz)
        self.order = max(1, int(order))
        if self.cutoff_hz <= 0.0:
            self.alpha = 1.0
        else:
            dt = 1.0 / self.fs_hz
            tau = 1.0 / (2.0 * np.pi * self.cutoff_hz)
            self.alpha = dt / (tau + dt)
        self.state = [0.0] * self.order
        self.initialized = False

    def update(self, x: float) -> float:
        x = float(x)
        if not self.initialized:
            self.state = [x] * self.order
            self.initialized = True
            return x
        y = x
        for i in range(self.order):
            self.state[i] = self.state[i] + self.alpha * (y - self.state[i])
            y = self.state[i]
        return float(y)


def apply_causal_lpf_series(x: np.ndarray, fs_hz: float, cutoff_hz: float, order: int) -> np.ndarray:
    lpf = _CausalLowPass(fs_hz, cutoff_hz, order)
    out = np.empty(len(x), dtype=np.float64)
    for i in range(len(x)):
        out[i] = lpf.update(float(x[i]))
    return out.astype(np.float32)


def run_inference(model: TCN, angle: np.ndarray, vel: np.ndarray, window_size: int, device: str) -> np.ndarray:
    n = int(min(len(angle), len(vel)))
    pred = np.zeros(n, dtype=np.float32)
    model.eval()
    with torch.no_grad():
        for t in range(n):
            start = max(0, t - window_size + 1)
            valid = t - start + 1
            x = np.zeros((2, window_size), dtype=np.float32)
            x[0, -valid:] = angle[start : t + 1]
            x[1, -valid:] = vel[start : t + 1]
            xt = torch.from_numpy(x).unsqueeze(0).to(device=device, dtype=torch.float32)
            y = model(xt)
            pred[t] = float(y[0, 0, -1].item())
    return pred


def corr(a: np.ndarray, b: np.ndarray) -> float:
    a = np.asarray(a, dtype=np.float64)
    b = np.asarray(b, dtype=np.float64)
    m = np.isfinite(a) & np.isfinite(b)
    a, b = a[m], b[m]
    if len(a) < 3:
        return float("nan")
    aa = a - np.mean(a)
    bb = b - np.mean(b)
    den = float(np.linalg.norm(aa) * np.linalg.norm(bb))
    if den == 0.0:
        return float("nan")
    return float(np.dot(aa, bb) / den)


# --- controller LPF settings (match deployed YAML) ---
_cfg = yaml.safe_load(CFG_PATH.read_text()) if CFG_PATH.is_file() else {}
_fs_cfg = int(_cfg.get("fs", 100))
_infer_hz = float(_cfg.get("infer_lpf_hz", 4.0))
_infer_ord = int(_cfg.get("infer_lpf_order", 4))
angle_lpf_hz = float(_cfg.get("angle_lpf_hz", _infer_hz))
angle_lpf_order = int(_cfg.get("angle_lpf_order", _infer_ord))
vel_lpf_hz = float(_cfg.get("vel_lpf_hz", _infer_hz))
vel_lpf_order = int(_cfg.get("vel_lpf_order", _infer_ord))
# Output LPF on .pt predictions (match cascade infer_out_lpf: default 6 Hz, 4th order)
out_lpf_hz = float(_cfg.get("out_lpf_hz", 6.0))
out_lpf_order = int(_cfg.get("out_lpf_order", 4))

# --- load log ---
data = np.load(FILE_PATH, allow_pickle=True)
time = np.asarray(data["time"], dtype=np.float64)
knee_angle = np.asarray(data["model_in_knee_angle_raw"], dtype=np.float32)
knee_vel = np.asarray(data["model_in_knee_vel_raw"], dtype=np.float32)
model_out_lpf = np.asarray(data["model_out_nmpkg"], dtype=np.float32)

if "model_out_nmpkg_raw" in data.files:
    model_out_raw = np.asarray(data["model_out_nmpkg_raw"], dtype=np.float32)
    has_raw = True
else:
    model_out_raw = None
    has_raw = False
    print(
        "[WARN] This .npz has no 'model_out_nmpkg_raw'. "
        "Re-record with an updated main_knee.py, or compare against model_out_nmpkg only."
    )

motor_torq = np.asarray(data["moment_raw"], dtype=np.float32)
n = min(len(time), len(knee_angle), len(knee_vel), len(model_out_lpf), len(motor_torq))
if has_raw:
    n = min(n, len(model_out_raw))
time = time[:n]
knee_angle = knee_angle[:n]
knee_vel = knee_vel[:n]
model_out_lpf = model_out_lpf[:n]
if has_raw:
    model_out_raw = model_out_raw[:n]
motor_torq = motor_torq[:n]

dt = float(np.median(np.diff(time))) if len(time) > 2 else 1.0 / _fs_cfg
fs_hz = float(1.0 / dt) if dt > 1e-6 else float(_fs_cfg)

# Causal LPF on inputs (as on device)
angle_f = apply_causal_lpf_series(knee_angle, fs_hz, angle_lpf_hz, angle_lpf_order)
vel_f = apply_causal_lpf_series(knee_vel, fs_hz, vel_lpf_hz, vel_lpf_order)

# --- checkpoint ---
print(f"[INFO] Loading {CKPT_PATH}")
ckpt = torch.load(str(CKPT_PATH), map_location="cpu", weights_only=False)
model_config = ckpt["model_config"]
mt = model_config.get("model_type", "tcn")
if mt != "tcn":
    raise ValueError(f"Checkpoint is model_type={mt!r}; this notebook only supports TCN.")

window_size = int(ckpt.get("window_size", _cfg.get("frame_length", 100)))
print(
    f"[INFO] window_size={window_size}  fs≈{fs_hz:.1f} Hz  "
    f"in LPF angle={angle_lpf_hz} Hz  vel={vel_lpf_hz} Hz  "
    f"out LPF={out_lpf_hz} Hz / {out_lpf_order}c"
)

device = "cuda" if torch.cuda.is_available() else "cpu"
model = TCN(**_tcn_ctor_kwargs(model_config)).eval()
model.load_state_dict(ckpt["model_state_dict"])
model.to(device)

pred_nmpkg_pt_raw = run_inference(model, angle_f, vel_f, window_size, device=device)
pred_nmpkg_pt = apply_causal_lpf_series(pred_nmpkg_pt_raw, fs_hz, out_lpf_hz, out_lpf_order)
print(
    f"[INFO] PyTorch Nm/kg raw (pre-output-LPF): min={pred_nmpkg_pt_raw.min():.4f} "
    f"max={pred_nmpkg_pt_raw.max():.4f}"
)
print(
    f"[INFO] PyTorch Nm/kg after {out_lpf_hz:g} Hz / {out_lpf_order}c output LPF: "
    f"min={pred_nmpkg_pt.min():.4f} max={pred_nmpkg_pt.max():.4f}"
)

# --- save sidecar npz ---
out_npz = str(Path(FILE_PATH).with_suffix("")) + "_pt_replay.npz"
save_kw = dict(
    time=time,
    pred_nmpkg_pt=pred_nmpkg_pt,
    pred_nmpkg_pt_raw=pred_nmpkg_pt_raw,
    out_lpf_hz=np.array(out_lpf_hz),
    out_lpf_order=np.array(out_lpf_order),
    model_in_knee_angle_raw=knee_angle,
    model_in_knee_vel_raw=knee_vel,
    model_in_knee_angle_lpf=angle_f,
    model_in_knee_vel_lpf=vel_f,
    model_out_nmpkg_logged=model_out_lpf,
    moment_raw=motor_torq,
    window_size=np.array([window_size]),
    ckpt_path=np.array(str(CKPT_PATH)),
)
if has_raw:
    save_kw["model_out_nmpkg_raw_logged"] = model_out_raw
np.savez(out_npz, **save_kw)
print(f"[INFO] Wrote replay arrays to {out_npz}")

if has_raw:
    print(
        f"[INFO] corr(PyTorch raw, model_out_nmpkg_raw) = "
        f"{corr(pred_nmpkg_pt_raw, model_out_raw):.4f}"
    )
    print(
        f"[INFO] corr(PyTorch out-LPF, logged model_out_nmpkg) = "
        f"{corr(pred_nmpkg_pt, model_out_lpf):.4f}"
    )
    print(f"[INFO] MAE(PyTorch raw vs TRT raw) = {np.mean(np.abs(pred_nmpkg_pt_raw - model_out_raw)):.6f} Nm/kg")
    print(
        f"[INFO] MAE(PyTorch out-LPF vs logged out-LPF) = "
        f"{np.mean(np.abs(pred_nmpkg_pt - model_out_lpf)):.6f} Nm/kg"
    )
else:
    print(f"[INFO] corr(PyTorch out-LPF, model_out_nmpkg) = {corr(pred_nmpkg_pt, model_out_lpf):.4f}")

PALETTE = {
    "angle": "#2196F3",
    "vel": "#FF9800",
    "pt": "#2E7D32",
    "raw": "#6A1B9A",
    "lfp": "#9C27B0",
    "torque": "#F44336",
}

[WARN] This .npz has no 'model_out_nmpkg_raw'. Re-record with an updated main_knee.py, or compare against model_out_nmpkg only.
[INFO] Loading /home/metamobility3/Jinwoo/os_kinetics/runs/0504_ik_id_knee_addbiomech_balanced/best_model.pt
[INFO] window_size=100  fs≈100.0 Hz  in LPF angle=6.0 Hz  vel=10.0 Hz  out LPF=6.0 Hz / 4c
[INFO] PyTorch Nm/kg raw (pre-output-LPF): min=-1.2146 max=1.2555
[INFO] PyTorch Nm/kg after 6 Hz / 4c output LPF: min=-0.9472 max=0.7441
[INFO] Wrote replay arrays to /home/metamobility3/Jinwoo/os_kinetics/knee_0506_gain0p12_ilseung_2_pt_replay.npz
[INFO] corr(PyTorch out-LPF, model_out_nmpkg) = 0.9956


In [ ]:
# Interactive time window (same pattern as analysis.ipynb)
out = widgets.Output()

slider = widgets.FloatRangeSlider(
    value=[float(time[0]), float(time[-1])],
    min=float(time[0]),
    max=float(time[-1]),
    step=0.1,
    description="Time (s):",
    layout=widgets.Layout(width="700px"),
    style={"description_width": "initial"},
    continuous_update=False,
    readout_format=".1f",
)


def _draw(t0, t1):
    mask = (time >= t0) & (time <= t1)
    t_sel = time[mask]

    fig, axs = plt.subplots(4, 1, figsize=(15, 12), sharex=True)

    axs[0].plot(t_sel, np.rad2deg(knee_angle[mask]), color=PALETTE["angle"], lw=1.2, label="raw (logged)")
    axs[0].plot(t_sel, np.rad2deg(angle_f[mask]), color=PALETTE["angle"], lw=1.0, ls="--", alpha=0.7, label="LPF → model")
    axs[0].set_ylabel("Knee angle (deg)", fontsize=13)
    axs[0].legend(fontsize=10, loc="upper right")
    axs[0].spines[["top", "right"]].set_visible(False)

    axs[1].plot(t_sel, np.rad2deg(knee_vel[mask]), color=PALETTE["vel"], lw=1.2, label="raw (logged)")
    axs[1].plot(t_sel, np.rad2deg(vel_f[mask]), color=PALETTE["vel"], lw=1.0, ls="--", alpha=0.7, label="LPF → model")
    axs[1].set_ylabel("Knee ang. vel. (deg/s)", fontsize=13)
    axs[1].legend(fontsize=10, loc="upper right")
    axs[1].spines[["top", "right"]].set_visible(False)
    axs[1].axhline(0, color="gray", lw=0.8, ls="--")

    axs[2].plot(
        t_sel, pred_nmpkg_pt[mask], color=PALETTE["pt"], lw=1.5,
        label=f"PyTorch ({out_lpf_hz:g}Hz/{out_lpf_order}c out LPF)",
    )
    axs[2].plot(
        t_sel, pred_nmpkg_pt_raw[mask], color=PALETTE["pt"], lw=0.9, ls="--", alpha=0.55,
        label="PyTorch raw (pre-output-LPF)",
    )
    if has_raw:
        axs[2].plot(t_sel, model_out_raw[mask], color=PALETTE["raw"], lw=1.2, alpha=0.85, label="logged model_out_nmpkg_raw")
    axs[2].plot(
        t_sel, model_out_lpf[mask], color=PALETTE["lfp"], lw=1.0, ls=":", alpha=0.9,
        label="logged model_out_nmpkg (output LPF)",
    )
    axs[2].set_ylabel("Model (Nm/kg)", fontsize=13)
    axs[2].legend(fontsize=10, loc="upper right")
    axs[2].spines[["top", "right"]].set_visible(False)
    axs[2].axhline(0, color="gray", lw=0.8, ls="--")

    axs[3].plot(t_sel, motor_torq[mask], color=PALETTE["torque"], lw=1.2, label="moment_raw (Nm)")
    axs[3].set_ylabel("Motor torque (Nm)", fontsize=13)
    axs[3].set_xlabel("Time (s)", fontsize=13)
    axs[3].legend(fontsize=10, loc="upper right")
    axs[3].spines[["top", "right"]].set_visible(False)
    axs[3].axhline(0, color="gray", lw=0.8, ls="--")

    fig.suptitle(f"{os.path.basename(FILE_PATH)}  |  ckpt: {CKPT_PATH.name}", fontsize=12, y=1.01)
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight", dpi=120)
    plt.close(fig)
    buf.seek(0)
    out.clear_output(wait=True)
    with out:
        display(Image(data=buf.read()))


def _on_slider(change):
    _draw(*slider.value)


slider.observe(_on_slider, names="value")
display(slider, out)
_draw(float(time[0]), float(time[-1]))

FloatRangeSlider(value=(0.010092153000186954, 300.1601306890002), continuous_update=False, description='Time (…

Output()

In [6]:
# Summary for the current slider window
t0_s, t1_s = slider.value
mask_s = (time >= t0_s) & (time <= t1_s)

rows = [
    (f"PyTorch out-LPF ({out_lpf_hz:g}Hz/{out_lpf_order}c)", pred_nmpkg_pt[mask_s]),
    ("PyTorch raw (pre-output-LPF)", pred_nmpkg_pt_raw[mask_s]),
    ("Logged model_out_nmpkg", model_out_lpf[mask_s]),
    ("Motor torque (Nm)", motor_torq[mask_s]),
]
if has_raw:
    rows.insert(2, ("Logged model_out_nmpkg_raw", model_out_raw[mask_s]))

print(f"Window {t0_s:.2f} – {t1_s:.2f} s")
print(f"{'Signal':<32} {'min':>10} {'max':>10} {'mean':>10} {'std':>10}")
print("-" * 74)
for name, sig in rows:
    print(f"{name:<32} {sig.min():10.4f} {sig.max():10.4f} {sig.mean():10.4f} {sig.std():10.4f}")

if has_raw:
    d_raw = pred_nmpkg_pt_raw[mask_s] - model_out_raw[mask_s]
    d_lpf = pred_nmpkg_pt[mask_s] - model_out_lpf[mask_s]
    print(
        f"\nΔ PyTorch raw − logged TRT raw: mean={d_raw.mean():.6f}  std={d_raw.std():.6f}  "
        f"max|Δ|={np.max(np.abs(d_raw)):.6f} Nm/kg"
    )
    print(
        f"Δ PyTorch out-LPF − logged out-LPF: mean={d_lpf.mean():.6f}  std={d_lpf.std():.6f}  "
        f"max|Δ|={np.max(np.abs(d_lpf)):.6f} Nm/kg"
    )

Window 0.01 – 300.16 s
Signal                                  min        max       mean        std
--------------------------------------------------------------------------
PyTorch out-LPF (6Hz/4c)            -0.9472     0.7441     0.0004     0.1335
PyTorch raw (pre-output-LPF)        -1.2146     1.2555     0.0003     0.1652
Logged model_out_nmpkg              -0.9460     0.7439     0.0004     0.1333
Motor torque (Nm)                   -7.8556     9.9902    -0.0040     1.4078
